Train ConvGamer V-JEPA foundation model on gameplay videos

ruff: noqa: E402 -- notebook cells import after setup cells by design.
This notebook installs [ConvGamer](https://github.com/Gabz4200/ConvGamer) from GitHub,
wires the JEPA stack through the repo composition root
(`convgamer.scripts.common` + `convgamer.training.engine`), and trains the full
backbone (`ConvGamerEncoder` in `models/convgamer`, `VJEPAPredictor`/`JEPALoss`
in `models/jepa`, wired by `ConvGamerVJEPAModel` in
`src/convgamer/modules/jepa_module.py`) on real gameplay MP4s.
The result is a self-supervised foundation encoder with no classification head.
A later notebook/script attaches an action-prediction head via
`encoder.add_classification_head(num_actions)`.

Prerequisites: on Kaggle, enable **Internet: ON** (right panel, for the clone
and the Hugging Face downloads) and pick a **GPU accelerator**
(P100 or T4 x2; CPU works for the smoke cells only). The run is time-boxed
to ~8.5h (`trainer.max_time`) with early stopping on the held-out loss, so
a single session covers it — checkpoints persist only via Save Version →
save outputs. All configured video datasets are public, so no HF token
is needed unless you add a gated one (then store it under Add-ons → Secrets).

Importance-weighted subsets: Terraria (~35 GB, 310 files) regularizes
generalization, so training takes a small strided slice (60 files); small
important repos repeat for balanced exposure (Cuphead ×5, Hollow Knight ×2).
Validation holds out 2 files per repo across all domains for the
early-stop signal — no repo is wasted on a whole-repo holdout.

Disk strategy (Kaggle /kaggle/working is ~20 GB): the dataset streams —
remote `.mp4` are listed (metadata only, zero bytes), fetched **one file at
a time just-in-time**, and deleted right after their sample when
`data.cleanup_downloads=true`. Disk stays O(1 file), never O(dataset), so
the 5-repo gameplay list trains without the ENOSPC crash. Never call
`_resolve_video_paths` on the full id list here — that snapshot-downloads
every repo up front.


In [ ]:
# Clone the repo (skip if already present) and add it to the path.
import os
import subprocess
import sys

if not os.path.isdir("ConvGamer"):
    if os.path.isdir(os.path.join(".", "src", "convgamer")):
        sys.path.insert(0, os.path.abspath("src"))
        print("repo: . (local checkout)")
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Gabz4200/ConvGamer.git"],
            check=True,
        )
        sys.path.insert(0, os.path.abspath("ConvGamer/src"))
        print("repo:", os.path.abspath("ConvGamer"))
else:
    sys.path.insert(0, os.path.abspath("ConvGamer/src"))
    print("repo:", os.path.abspath("ConvGamer"))

In [ ]:
# Install runtime deps (torch/torchvision ship with the Kaggle image, so they
# are intentionally not listed; CPU-only runtimes still run the smoke cells).
# !pip install -q hydra-core omegaconf pytorch-lightning transformers
# !pip install -q huggingface-hub einops decord pillow

In [ ]:
import torch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 1. Compose the JEPA training config

`model=jepa` requires `data=jepa` and vice versa (`build_datamodule` fails
fast otherwise); trainer/optimizer groups stay freely composable.
`data.mode=video` trains on `video_dataset_ids` below (`mode=fixture` is the
offline smoke clip only). Kaggle-sized for 2x T4: the full 5-repo gameplay
list from `configs/data/jepa.yaml` (~2k clips), batch 8 per GPU
(effective 8 x 2 GPUs x 2 accum = 32; drop to 4 per GPU for effective 16
if VRAM OOMs), `accelerator/devices/strategy` all `auto`
(→ cuda/2/DDP on the T4 host, cpu/1/single elsewhere).
For the full run, raise `MAX_STEPS` (paper Appendix A uses 135000).
Default 20000 ≈ 290 passes over the ~2170-sample weighted pool; early
stopping (`trainer.early_stopping`, patience 15 val checks) and the 8.5h
`trainer.max_time` backstop end the run sooner on convergence or timeout.
LR warmup rescales with the budget (paper 12k/135k ≈ 9% → 1500/20000).

Streaming flags below are the disk fix: `data.streaming=true` lists remote
files without downloading and fetches one `.mp4` at a time inside the
IterableDataset; `data.cleanup_downloads=true` deletes each file after its
sample and evicts the scoped Hub cache. `num_workers=2` bounds concurrent
just-in-time downloads (4 workers × full snapshots was part of the OOM).


In [ ]:
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

from convgamer.scripts.common import CONFIG_DIR

USE_CUDA = torch.cuda.is_available()
MAX_STEPS = 20000  # ~290 passes over the weighted pool; early stop ends sooner
BATCH_SIZE = 8  # per-GPU (effective 8 x 2 GPUs x 2 accum = 32); drop to 4 on OOM
WARMUP_STEPS = 1500  # ~9% of MAX_STEPS, matching the paper's 12k/135k ratio
LAMBDA_WARMUP = round(MAX_STEPS * 30000 / 135000)  # same 22% ratio as paper epochs 50-100
# Keep every downloaded byte inside the writable working disk (Kaggle or CWD).
DATA_DIR = "/kaggle/working/data/jepa" if os.path.isdir("/kaggle/working") else "./data/jepa"
os.makedirs(DATA_DIR, exist_ok=True)
os.environ["HF_HUB_CACHE"] = os.path.join(DATA_DIR, ".hf_global_cache")

overrides = [
    "model=jepa",
    "data=jepa",
    "trainer=jepa",
    "optimizer=jepa",
    "data.mode=video",
    f"trainer.precision={'16-mixed' if USE_CUDA else '32-true'}",
    f"trainer.max_steps={MAX_STEPS}",
    f"model.warmup_steps={WARMUP_STEPS}",
    f"model.loss.lambda_warmup_steps={LAMBDA_WARMUP}",
    f"data.batch_size={BATCH_SIZE}",
    "data.num_workers=2",
    f"data.data_dir={DATA_DIR}",
    # Streaming disk fix (see header): O(1-file) disk instead of O(dataset).
    "data.streaming=true",
    "data.cleanup_downloads=true",
]

with initialize_config_dir(config_dir=str(CONFIG_DIR), version_base=None):
    cfg = compose(config_name="config", overrides=overrides)

print(OmegaConf.to_yaml(cfg.data))
print("video ids:", len(cfg.data.video_dataset_ids))

## 2. Sampling table (no bytes), then smoke-test one streaming batch

No upfront snapshot download: `describe_sampling` lists remote files
(filenames, not blobs) and reports each repo's total → train pool → repeat
→ effective samples, so the importance weighting is visible before any
byte moves. The first training batch then pulls just enough `.mp4`
just-in-time (~8 files for batch 8) through the streaming
`GameVideoDataset`. `build_model` dispatches on the `convgamer.models.jepa`
marker to `ConvGamerVJEPAModel` (encoder/predictor/loss from
`configs/model/jepa.yaml`). Batches are `(context, target, mask)` triplets
of standardized RGB.


In [ ]:
from convgamer.data.sources import describe_sampling, disk_free_bytes
from convgamer.scripts.common import build_datamodule, build_model

print(f"free disk in {DATA_DIR}: {disk_free_bytes(DATA_DIR) / 1e9:.2f} GB")
try:
    rows = describe_sampling(
        list(cfg.data.video_dataset_ids),
        caps=dict(cfg.data.get("subset_caps", {})),
        repeats=dict(cfg.data.get("oversample", {})),
        val_tail=int(cfg.data.get("val_tail", 2)),
    )
except Exception as error:
    print(f"remote listing failed ({error}); continuing without the table")
    rows = []
if rows:
    print(f"{'repo':50s} {'total':>6s} {'train':>6s} {'xN':>3s} {'effective':>9s}")
    for repo, total, pool, rep, eff in rows:
        print(f"{repo:50s} {total:6d} {pool:6d} {rep:3d} {eff:9d}")
    print(
        f"{'TOTAL':50s} {sum(r[1] for r in rows):6d} {sum(r[2] for r in rows):6d} {'':>3s} "
        f"{sum(r[4] for r in rows):9d} samples/pass"
    )

model = build_model(cfg)
datamodule = build_datamodule(cfg)
datamodule.setup("fit")

train_loader = datamodule.train_dataloader()
batch = next(iter(train_loader))
x, y, mask = batch
# Release the probe iterator so the streaming worker's just-in-time file can
# be cleaned before fit() spawns its own loader set (avoids double-fetch).
del train_loader
print(
    "x:", tuple(x.shape), x.dtype, "| y:", tuple(y.shape), "| mask:", tuple(mask.shape), mask.dtype
)
print(f"free disk after smoke batch: {disk_free_bytes(DATA_DIR) / 1e9:.2f} GB")

with torch.no_grad():
    loss = model(x[:1], y[:1], mask[:1])
print("smoke loss:", float(loss))

## 3. Smoke-fit (a few optimizer steps, verifies training loop + EMA)


In [ ]:
import pytorch_lightning as pl

from convgamer.training.engine import create_trainer

pl.seed_everything(int(cfg.get("seed", 42)), workers=True)
smoke_cfg = OmegaConf.merge(
    cfg,
    {
        "trainer": {
            "max_steps": 5,
            "limit_val_batches": 0,
            "min_steps": 0,
            "early_stopping": {"enabled": False},
            "collapse_guard": {"enabled": False},
        }
    },
)
smoke_trainer = create_trainer(smoke_cfg)  # type: ignore[arg-type]
smoke_trainer.fit(model, datamodule=datamodule)
print("smoke fit ok; ema shadow device:", next(model.ema_encoder.shadow_module.parameters()).device)

## 4. LR finder, then the full JEPA pretraining run

Lightning sweeps the LR exponentially, restores the pre-sweep weights, and
writes the suggestion back to `model.lr` (read by `configure_optimizers`).
The run ends whichever comes first: `MAX_STEPS`, early stopping on
`val/loss` (15 checks ≈ 1500 steps without improvement, after a 2000-step
floor so the weighted pool is seen ~30× first), the collapse guard
(`CollapseGuardCallback` stops on 3 consecutive ~zero `val/pred_std`
checks — JEPA loss alone cannot detect representation collapse), or the
8.5h `max_time` backstop. Best weights survive via top-3 `val/loss`
checkpointing. The loss adds LeJEPA SIGReg isotropy pressure
(`loss.sigreg_lambda=0.02`, the paper's goto value; logged as
`train/sigreg`): predicted latents are regularized toward an isotropic
Gaussian, which provably minimizes downstream risk and resists collapse
without any teacher swap — EMA stays. A linear probe would be the ideal
stop signal, but no dataset carries per-clip action labels (Terraria has
provenance metadata only; others are bare MP4s), so the frozen-schedule
`val/loss` + feature std sentinel is the label-free substitute.
Training streams file-by-file (download → train → delete, with scoped-cache
eviction), so the working disk stays flat for the whole run. The train
loader pins memory, persists workers across epochs (keeps the per-sample
mask salt unique on every pass), and prefetches to cover JIT-download
stalls. Checkpoints land in `checkpoints/jepa/`
(see `configs/trainer/jepa.yaml`). On Kaggle, /kaggle/working persists only
via Save Version → save outputs.
Resume with `trainer.fit(model, datamodule=datamodule, ckpt_path="...")`.

Tight-disk alternative (same building blocks, explicit chunks): iterate
`iter_video_chunks(ids, DATA_DIR, chunk_size=30)`, build a datamodule per
chunk from the returned local paths, `fit` a slice of steps, then
`cleanup_chunk(chunk_paths, DATA_DIR)` before the next chunk.


In [ ]:
from convgamer.training.engine import maybe_run_lr_finder

pl.seed_everything(int(cfg.get("seed", 42)), workers=True)
trainer = create_trainer(cfg)

cfg = OmegaConf.merge(
    cfg,
    {
        "trainer": {
            "lr_finder": {
                "enabled": True,
                "min_lr": 1e-6,
                "max_lr": 1e-2,
                "num_training": 50,
                "mode": "exponential",
            }
        }
    },
)
try:
    maybe_run_lr_finder(trainer, model, datamodule, cfg)  # type: ignore[arg-type]
except Exception as error:  # tuner is flaky under DDP; keep the config LR
    print(f"lr finder failed ({error}); keeping config lr {model.lr}")
print("training with lr:", model.lr)
print(f"free disk before fit: {disk_free_bytes(DATA_DIR) / 1e9:.2f} GB")
trainer.fit(model, datamodule=datamodule)
print(f"free disk after fit: {disk_free_bytes(DATA_DIR) / 1e9:.2f} GB")

In [ ]:
# List checkpoints produced by this run.
import glob

ckpt_dir = str(cfg.trainer.get("checkpoint_callback", {}).get("dirpath", "checkpoints/jepa"))
for ckpt in sorted(glob.glob(f"{ckpt_dir}/*.ckpt")):
    print(ckpt)

## 5. Next step: action-prediction head (for the follow-up task)

The checkpoint holds a headless foundation encoder. Attach a head later:
`encoder.add_classification_head(num_actions)` then fine-tune supervised on
labeled gameplay frames. Skeleton (not run here):

Future label source: `markov-ai/gaming-500-hours` reportedly ships
per-workflow event logs + `metadata.jsonl` game labels (verify with
`list_repo_files` before relying on it) — the closest thing to action
supervision in the pool (Terraria has provenance only).


In [ ]:
# from convgamer.modules.jepa_module import ConvGamerVJEPAModel
#
# CKPT = "checkpoints/jepa/last.ckpt"  # or a jepa-convgamer-*.ckpt
# NUM_ACTIONS = 8  # set to your action space
# jepa = ConvGamerVJEPAModel.load_from_checkpoint(CKPT, strict=False)
# encoder = jepa.encoder
# encoder.add_classification_head(NUM_ACTIONS)
# for p in encoder.frame_encoder.parameters():
#     p.requires_grad = False  # optional: linear-probe the head first
# print(type(encoder.head), "| frozen frame encoder ready for action fine-tuning")